# Colab GPU로 RAG 임베딩 만들기

이 노트북은 `chunk.7z`를 업로드해 압축을 풀고, `chunks.jsonl` 구조를 검증한 뒤 BGE-M3 GPU 임베딩을 수행합니다. 마지막 셀에서 생성된 `rag.sqlite3`를 다운로드합니다.

실행 전에 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU(또는 GPU)** 를 선택하세요. 입력 압축파일은 `chunk/네이버뉴스/{id}/chunks.jsonl`처럼 최상위 `chunk/`를 포함해도 되고, `네이버뉴스/{id}/chunks.jsonl`처럼 소스 디렉터리부터 시작해도 됩니다. `info.json`은 선택 파일입니다.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/2026WAI/embedding-datas.git"
REPO_DIR = Path("/content/embedding-datas")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt"), "py7zr>=0.21.0"],
    check=True,
)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU를 찾지 못했습니다. Colab 런타임 유형을 GPU로 바꾼 뒤 처음부터 다시 실행하세요.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import files

uploaded = files.upload()  # chunk.7z 하나를 선택합니다.
archive_names = [name for name in uploaded if name.lower().endswith(".7z")]
if len(archive_names) != 1:
    raise ValueError(".7z 파일을 정확히 하나 업로드하세요.")
ARCHIVE_PATH = Path(archive_names[0]).resolve()
print(f"업로드 완료: {ARCHIVE_PATH.name} ({ARCHIVE_PATH.stat().st_size / 1024**3:.2f} GiB)")

In [ ]:
import py7zr
import shutil

EXTRACT_DIR = Path("/content/uploaded_chunks")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir()

with py7zr.SevenZipFile(ARCHIVE_PATH, mode="r") as archive:
    names = archive.getnames()
    unsafe = [name for name in names if Path(name).is_absolute() or ".." in Path(name).parts]
    if unsafe:
        raise ValueError(f"안전하지 않은 압축 경로가 있습니다: {unsafe[:3]}")
    archive.extractall(path=EXTRACT_DIR)

print(f"압축 해제 완료: 파일 {len(names):,}개")

In [ ]:
from collections import Counter

# archive에 chunk/가 있으면 그 안을, 없으면 압축 해제 루트를 입력으로 사용합니다.
CHUNK_ROOT = EXTRACT_DIR / "chunk" if (EXTRACT_DIR / "chunk").is_dir() else EXTRACT_DIR
chunk_files = sorted(CHUNK_ROOT.rglob("chunks.jsonl"))
json_files = sorted(CHUNK_ROOT.rglob("chunks.json"))
info_files = sorted(CHUNK_ROOT.rglob("info.json"))

if not chunk_files:
    if json_files:
        raise ValueError("chunks.json 파일만 발견했습니다. 현재 동기화 도구의 입력 형식은 JSON Lines인 chunks.jsonl입니다.")
    raise FileNotFoundError("chunks.jsonl을 찾지 못했습니다. 압축 안의 디렉터리 구조를 확인하세요.")

sources = Counter(path.relative_to(CHUNK_ROOT).parts[0] for path in chunk_files)
print(f"입력 루트: {CHUNK_ROOT}")
print(f"chunks.jsonl: {len(chunk_files):,}개, info.json: {len(info_files):,}개")
print("소스별 chunks.jsonl 수:")
for source, count in sources.most_common():
    print(f"  - {source}: {count:,}")
print("예시:", chunk_files[0].relative_to(CHUNK_ROOT))

In [ ]:
# 처음 실행은 BGE-M3 모델을 내려받습니다. 이후 같은 Colab 세션에서는 재사용됩니다.
subprocess.run([sys.executable, "download_model.py"], cwd=REPO_DIR, check=True)

In [ ]:
# T4 16 GB 기준으로 안전한 시작값입니다. CUDA 메모리가 충분하면 64로 올릴 수 있습니다.
BATCH_SIZE = 32
DB_PATH = REPO_DIR / "vector_store" / "rag.sqlite3"

subprocess.run(
    [
        sys.executable, "sync_embeddings.py", "--rebuild",
        "--chunk-dir", str(CHUNK_ROOT),
        "--db-path", str(DB_PATH),
        "--device", "cuda",
        "--batch-size", str(BATCH_SIZE),
    ],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"생성된 DB가 없습니다: {DB_PATH}")
sys.path.insert(0, str(REPO_DIR))
from rag_store import connect_database

connection = connect_database(DB_PATH)
try:
    chunk_count = connection.execute("SELECT COUNT(*) FROM chunks").fetchone()[0]
    vector_count = connection.execute("SELECT COUNT(*) FROM vec_chunks").fetchone()[0]
    connection.execute("PRAGMA wal_checkpoint(TRUNCATE)")
finally:
    connection.close()

if chunk_count != vector_count:
    raise RuntimeError(f"청크({chunk_count:,})와 벡터({vector_count:,}) 수가 다릅니다.")
print(f"검증 완료: 청크/벡터 {chunk_count:,}개")
print(f"DB 크기: {DB_PATH.stat().st_size / 1024**3:.2f} GiB")

In [ ]:
files.download(str(DB_PATH))